In [ ]:
# Run this only once if the libraries are not installed
#!pip install pyarrow tensorflow scikit-learn matplotlib seaborn

In [1]:
#Import library
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

In [2]:
#If this cell fails you need to change the runtime of your colab notebook to GPU
# Go to Runtime -> Change Runtime Type and select GPU
assert torch.cuda.is_available(), "GPU is not enabled"

#  use gpu if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
# Load parquet file
df_aps = pd.read_parquet('meme_clean.parquet')

print(df_aps.shape)
df_aps.head()

(10470122, 21)


,associated_device_name,band,health,speed,maxspeed,signal_db,signal_strength,snr,last_connection_time,snapshot_ts,...,hour,date,signal_integrity,signal_score,performance,swarm_name,cpu_utilization,mem_usage,client_count,overloaded
0,AP-CEDU26,5.0,100.0,96.0,192.0,-50.0,5.0,46.0,1.773452e+12,2026-03-16 17:31:51,...,17,2026-03-16,Excellent++,1.000000,Fair,AP-CEDU26,8.0,65.548160,3.0,False
1,AP-CCOM31,5.0,90.0,156.0,192.0,-55.0,5.0,41.0,1.773682e+12,2026-03-16 17:31:51,...,17,2026-03-16,Excellent,0.939167,Good,AP-CCOM31,14.0,66.193870,50.0,False
2,AP-ETSE11,5.0,96.0,173.0,192.0,-48.0,5.0,48.0,1.773669e+12,2026-03-16 17:31:51,...,17,2026-03-16,Excellent++,0.984000,Excellent,AP-ETSE11,9.0,68.488426,7.0,False
3,AP-CEDU33,5.0,100.0,96.0,96.0,-48.0,5.0,48.0,1.773681e+12,2026-03-16 17:31:51,...,17,2026-03-16,Excellent++,1.000000,Excellent,AP-CEDU33,28.0,70.271943,57.0,True
4,AP-CEDU12,5.0,100.0,96.0,96.0,-63.0,4.0,33.0,1.773681e+12,2026-03-16 17:31:51,...,17,2026-03-16,Excellent,0.945833,Excellent,AP-CEDU12,11.0,69.888112,40.0,False


In [3]:
#for now, drop nan
df_aps = df_aps.dropna()

In [4]:
# Convert timestamp
df_aps['timestamp'] = pd.to_datetime(df_aps['timestamp'])

# Sort values
df_aps_sorted = df_aps.sort_values(['associated_device_name', 'timestamp'])

print(df_aps[['associated_device_name', 'timestamp']].head())

  associated_device_name           timestamp
0              AP-CEDU26 2026-03-14 01:37:50
1              AP-CCOM31 2026-03-16 17:22:08
2              AP-ETSE11 2026-03-16 13:48:52
3              AP-CEDU33 2026-03-16 17:12:23
4              AP-CEDU12 2026-03-16 17:11:54


In [5]:
#Numerical features
features = ['signal_score', 'signal_strength', 'signal_db', 'snr', 'cpu_utilization', 'mem_usage',
    'client_count', 'health', 'speed', 'maxspeed']

#Relation between name and time stamps
required_columns = ['associated_device_name', 'timestamp'] + features

model_df = df_aps_sorted[required_columns].copy()

model_df.head()

,associated_device_name,timestamp,signal_score,signal_strength,signal_db,snr,cpu_utilization,mem_usage,client_count,health,speed,maxspeed
8043130,48:00:20:C0:BA:7E,2026-04-08 07:30:05.083,0.850833,3.0,-67.0,25.0,8.0,59.813102,2.0,95.0,86.0,192.0
8163347,48:00:20:C0:BA:7E,2026-04-08 09:49:44.000,0.759500,3.0,-71.0,21.0,7.0,60.975170,2.0,88.0,12.0,1200.0
8188186,48:00:20:C0:BA:7E,2026-04-08 09:49:44.000,1.000000,5.0,-48.0,44.0,10.0,61.915797,4.0,100.0,172.0,286.0
8195809,48:00:20:C0:BA:7E,2026-04-08 09:49:44.000,0.823333,3.0,-70.0,22.0,7.0,62.316641,6.0,100.0,360.0,1200.0
8203438,48:00:20:C0:BA:7E,2026-04-08 09:49:44.000,0.886667,3.0,-66.0,26.0,8.0,61.158618,7.0,100.0,288.0,1200.0


In [6]:
# Check missing values
print(model_df.isnull().sum())

# Forward fill per AP
model_df = model_df.groupby('associated_device_name').apply(
    lambda x: x.ffill().bfill()
).reset_index(drop=True)

print(model_df.isnull().sum())

associated_device_name    0
timestamp                 0
signal_score              0
signal_strength           0
signal_db                 0
snr                       0
cpu_utilization           0
mem_usage                 0
client_count              0
health                    0
speed                     0
maxspeed                  0
dtype: int64
associated_device_name    0
timestamp                 0
signal_score              0
signal_strength           0
signal_db                 0
snr                       0
cpu_utilization           0
mem_usage                 0
client_count              0
health                    0
speed                     0
maxspeed                  0
dtype: int64


In [ ]:
#Group the data by hours so we can predict based on hours
hourly_df = (model_df.set_index('timestamp').groupby('associated_device_name').resample('1H').mean(numeric_only=True).reset_index())

hourly_df.head()

In [ ]:
#Number of hours of data of each ap after the transformations
ap_counts = (hourly_df['associated_device_name'].value_counts())

print(ap_counts.head)

In [ ]:
# Select one AP
ap_name = hourly_df['associated_device_name'].iloc[10]

ap_df = hourly_df[hourly_df['associated_device_name'] == ap_name].copy()

print(f'Selected AP: {ap_name}')
print(ap_df.shape)

ap_df

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(ap_df['timestamp'], ap_df['signal_score'])
plt.title(f'Signal Score Over Time — {ap_name}')
plt.xlabel('Time')
plt.ylabel('Signal Score')
plt.grid(True)
plt.show()

#Train a nn

In [ ]:
#Neural networks work better with normalized values.

scaler = MinMaxScaler()

scaled_data = scaler.fit_transform(ap_df[features])

scaled_df = pd.DataFrame(scaled_data, columns=features)

scaled_df.head()

We will create a predictor that, based on the past 24 hours, predicts the next 5. So, we create:

*   Input window = previous 24 hours
*   Output horizon = next 5 hours

In other words:

* X → last 24 hours
* y → next 5 values of signal_score

In [ ]:
WINDOW_SIZE = 24
FORECAST_HORIZON = 5

TARGET_COLUMN = 'signal_score'

target_idx = features.index(TARGET_COLUMN)


def create_sequences(data, window_size, forecast_horizon, target_idx):
    X = []
    y = []


    for i in range(len(data) - window_size - forecast_horizon):
      X.append(data[i:i+window_size])
      y.append(data[i+window_size:i+window_size+forecast_horizon, target_idx])

    return np.array(X), np.array(y)


X, y = create_sequences(scaled_data, WINDOW_SIZE, FORECAST_HORIZON, target_idx)

print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
#Check if the the X has samples
print(ap_df.shape)

print(len(scaled_data))

print(len(scaled_data) - WINDOW_SIZE - FORECAST_HORIZON)

##training

In [ ]:
#Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(X_train.shape)
print(X_test.shape)

In [ ]:
model = Sequential([LSTM(64, return_sequences=True, input_shape=(WINDOW_SIZE, len(features))),

    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(FORECAST_HORIZON)])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(X_train, y_train, validation_split=0.2, epochs=50, batch_size=32, callbacks=[early_stop])

##Evaluation

In [ ]:
results = model.evaluate(X_test, y_test)

print(f'Test Loss: {results[0]}')
print(f'Test MAE: {results[1]}')

###Prediction

In [ ]:
predictions = model.predict(X_test)

print(predictions.shape)

In [ ]:
#We compare the prediction for the next hour.
plt.figure(figsize=(15,5))

plt.plot(
    y_test[:,0],
    label='Real'
)

plt.plot(
    predictions[:,0],
    label='Predicted'
)

plt.title('Next-Hour Signal Score Prediction')
plt.xlabel('Samples')
plt.ylabel('Signal Score')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#We use the latest available window to predict the next 5 hours
last_window = scaled_data[-WINDOW_SIZE:]

last_window = np.expand_dims(last_window, axis=0)

future_prediction = model.predict(last_window)

future_prediction = future_prediction[0]

print('Predicted Signal Score for the Next 5 Hours:')

for i, value in enumerate(future_prediction, start=1):
    print(f'Hour +{i}: {value:.4f}')

In [ ]:
#Convert Predicted Scores into Signal Integrity Labels
def score2integrity(score):

    if score >= 0.95:
        return 'Excellent++'

    elif score >= 0.90:
        return 'Excellent+'

    elif score >= 0.80:
        return 'Excellent'

    elif score >= 0.70:
        return 'Good'

    elif score >= 0.50:
        return 'Fair'

    else:
        return 'Poor'

future_labels = [score2integrity(x) for x in future_prediction]

for i, label in enumerate(future_labels, start=1):
    print(f'Hour +{i}: {label}')